# LCEL (LangChain Expression Language)

In [1]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_learning.llm import get_llm

llm = get_llm()

**术语：**
- **LCEL** = LangChain Expression Language（LangChain 表达式语言），用 `|` 符号把组件串起来
- **Runnable** = "可运行组件"，LCEL 里所有东西都是 Runnable，可以用 `|` 连接
- **Prompt** = 提示词模板
- **Parser** = 解析器，处理 LLM 的输出

```mermaid
graph TD
    A[可运行组件<br/>Runnable] --> B["串联 | "]
    A --> C[并行合并]
    A --> D["透传 / 转换"]
    B --> E["Prompt | LLM | Parser"]
    C --> F[RunnableParallel]
    D --> G[RunnablePassthrough]
    D --> H[RunnableLambda]
    style A fill:#FFF3E0,stroke:#E65100
    style B fill:#E3F2FD,stroke:#1565C0
    style C fill:#E3F2FD,stroke:#1565C0
    style D fill:#E3F2FD,stroke:#1565C0
```

### 1. 基础链

In [2]:
chain = ChatPromptTemplate.from_template("给主题起三个好标题：{topic}") | llm | StrOutputParser()
print(chain.invoke({"topic": "AI 编程助手"}))

针对“AI编程助手”这一主题，我为你构思了三个不同侧重点的标题，分别适用于技术深度、效率提升和未来趋势三个方向：

1.  **《代码的“第二大脑”：AI编程助手如何重塑开发者的思维模式》**
    *   **侧重点：** 认知与协作。这个标题将AI定位为开发者的“外脑”，探讨人机协作如何改变编程的思考方式，适合深度分析或体验类文章。

2.  **《从“写代码”到“审代码”：AI助手如何让开发者效率翻倍》**
    *   **侧重点：** 效率与工作流。通过“写”与“审”的对比，突出AI从辅助生成到质量把控的角色转变，适合教程、评测或效率提升类内容。

3.  **《代码的“新语法”：当AI成为你的结对编程伙伴》**
    *   **侧重点：** 未来趋势与协作模式。借用“结对编程”这一经典概念，暗示AI不再是工具，而是平等的“伙伴”，适合探讨行业变革或前瞻性文章。

**建议：** 如果你的文章更偏向**技术实操**，推荐标题2；如果偏向**行业思考**或**个人体验**，标题1和3会更有深度和吸引力。


**术语：**
- **基础链** = `PromptTemplate | LLM | Parser` 是最简单的链
- **`|` 运算符** = 类似 Unix 管道，前一个的输出自动传给后一个

```python
# 这条链的意思是:
# 模板生成提示词 → LLM 生成回复 → 解析器提取文本
chain = prompt | llm | parser
```

### 2. RunnablePassthrough 透传

In [3]:
chain = (
    {"topic": RunnablePassthrough()}
    | ChatPromptTemplate.from_template("写一句关于{topic}的话")
    | llm | StrOutputParser()
)
print(chain.invoke("人工智能"))

人工智能是人类的镜像，映照出我们最深的智慧与恐惧，却永远无法复制那束名为“意识”的微光。


**术语：**
- **RunnablePassthrough** = "透传"，把输入原样传下去，常用于把字符串包装成字典
- **字典 / dict** = `{"key": value}` 形式的 Python 数据结构

```mermaid
graph LR
    A["输入: '人工智能'"] --> B[RunnablePassthrough]
    B --> C["包装成 {topic: '人工智能'}"]
    C --> D[PromptTemplate]
    D --> E[LLM]
    E --> F[StrOutputParser]
    style B fill:#E8F5E9,stroke:#2E7D32
```

### 3. 并行执行

In [4]:
chain1 = ChatPromptTemplate.from_template("用一句话总结：{topic}") | llm | StrOutputParser()
chain2 = ChatPromptTemplate.from_template("列出{topic}的三个关键点") | llm | StrOutputParser()

parallel = RunnableParallel(summary=chain1, keypoints=chain2)
result = parallel.invoke({"topic": "RAG 技术"})
print("总结:", result["summary"][:100])
print("要点:", result["keypoints"][:200])

总结: RAG（检索增强生成）技术通过先检索外部知识库中的相关信息，再将其作为上下文输入给大语言模型，从而有效提升生成内容的准确性、时效性和可解释性。
要点: RAG（检索增强生成）技术的三个关键点如下：

1.  **检索（Retrieval）**：这是RAG的核心环节。它负责从外部知识库（如文档、数据库、网页）中，根据用户的输入查询，快速、准确地找到最相关的信息片段。这一步通常依赖于**向量数据库**和**语义搜索**技术，将文本转化为向量并计算相似度，从而召回与问题最匹配的上下文。

2.  **增强（Augmentation）**：将检索到的相关


**术语：**
- **RunnableParallel** = "并行运行"，多条链同时执行，互不等待
- **并行** = 同时做多件事，提高效率
- **参数名 = 结果 key**：`summary=chain1` 表示 chain1 的结果存到 `result["summary"]`

```python
parallel = RunnableParallel(summary=chain1, keypoints=chain2)
result = parallel.invoke(...)
# result["summary"]   → chain1 的输出
# result["keypoints"] → chain2 的输出
```

```mermaid
graph LR
    A[输入] --> B[RunnableParallel]
    B --> C[总结链]
    B --> D[关键点链]
    C --> E["result.summary"]
    D --> F["result.keypoints"]
    style B fill:#E3F2FD,stroke:#1565C0
    style C fill:#FFF3E0,stroke:#E65100
    style D fill:#FFF3E0,stroke:#E65100
```

### 4. RunnableLambda 注入自定义函数

In [5]:
def word_count(text: str) -> dict:
    return {"chars": len(text), "words": len(text.split())}

chain = (
    ChatPromptTemplate.from_template("写一句关于{topic}的话")
    | llm | StrOutputParser()
    | RunnableLambda(word_count)
)
print(chain.invoke({"topic": "机器学习"}))

{'chars': 24, 'words': 1}


**术语：**
- **RunnableLambda** = 把普通 Python 函数包装成 Runnable，嵌入 LCEL 链
- **Lambda** = 这里指"函数"，不是 Python 的 lambda 关键字

`word_count` 函数接收 LLM 输出的文本，计算字数和词数，返回字典。

### 5. 批处理

In [6]:
chain = ChatPromptTemplate.from_template("介绍{topic}（一句话）") | llm | StrOutputParser()
results = chain.batch([{"topic": "Python"}, {"topic": "Rust"}])
for i, r in enumerate(results, 1):
    print(f"{i}. {r}")

1. Python是一种简洁易读、功能强大的高级编程语言，广泛应用于数据分析、人工智能、Web开发与自动化等领域。
2. Rust是一种注重性能、安全性和并发性的系统编程语言，通过所有权和借用检查机制在编译时消除内存错误，无需垃圾回收即可实现高效运行。


**术语：**
- **batch** = "批处理"，一次传入多个输入，逐个执行链
- **批量** = 同时处理多条数据
- **max_concurrency** = 最大并发数，控制同时跑多少个请求

`.batch()` 方法批量处理多个输入，底层自动管理并发，返回结果列表。适合需要对多组数据执行相同链的场景。

```python
# 默认全部并发
results = chain.batch([input1, input2, input3])

# 限制同时最多 2 个
results = chain.batch([input1, input2, input3], config={"max_concurrency": 2})
```